In [ ]:
# === Setup ===
# Runtime: <2m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
import torch
torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
DEVICE = torch.device("cuda" if RUNTIME_PROFILE == "gpu" else "cpu")
if RUNTIME_PROFILE == "gpu" and not torch.cuda.is_available(): raise RuntimeError("GPU profile requested but CUDA is unavailable")
torch.set_default_device(DEVICE)
_device_probe = (torch.ones(8, device=DEVICE) @ torch.ones(8, device=DEVICE)).item()
print(f"Compute device: {DEVICE}; probe={_device_probe:.1f}")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Experiment — learning rate

**Hypothesis:** learning rate quá nhỏ học chậm; quá lớn thiếu ổn định.

In [ ]:
def train(lr):
    torch.manual_seed(42); X=torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]]); y=torch.tensor([[0.],[1.],[1.],[0.]])
    m=torch.nn.Sequential(torch.nn.Linear(2,8),torch.nn.Tanh(),torch.nn.Linear(8,1)); o=torch.optim.SGD(m.parameters(),lr=lr); f=torch.nn.BCEWithLogitsLoss()
    for _ in range(100 if FAST_MODE else 500): o.zero_grad(); loss=f(m(X),y); loss.backward(); o.step()
    return loss.item()
results={lr:train(lr) for lr in (.001,.1,2.)}; print(results)

**Observation:** kết luận dựa trên loss đo được; không suy ra một learning rate tối ưu cho mọi optimizer/dataset.